# 스마트 창고 출고 지연 예측 v7

### v6 → v7 핵심 개선사항
**v6 문제 진단**: CB Optuna 7시간+ 병목, test 고부하 과소예측, OOF-Public 갭 1.37

1. **CB Optuna 제거** → 좋은 기본값 하드코딩 (7시간 절약)
2. **Adversarial Sample Weighting** → train/test 분포 차이 직접 보정 (가장 핵심)
3. **Tail 보정 강화** → test 고부하 패턴(order_inflow 40% 높음) 반영
4. **Multi-seed 3→5** → 분산 감소
5. **ExtraTrees** → 그래디언트 부스팅과 다른 다양성


In [1]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import optuna
import warnings

from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor
from xgboost import XGBRegressor
from sklearn.ensemble import ExtraTreesRegressor, RandomForestClassifier
from sklearn.model_selection import GroupKFold, KFold, cross_val_predict
from sklearn.metrics import mean_absolute_error
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from scipy import stats
from scipy.optimize import minimize

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

SEED   = 42
N_FOLDS = 5
TARGET = 'avg_delay_minutes_next_30m'
ID_COLS = ['ID', 'layout_id', 'scenario_id']
POWER  = 0.4
np.random.seed(SEED)
print('환경 설정 완료')


환경 설정 완료


c:\Users\user\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. 데이터 로드 + 레이아웃 처리

In [2]:
train  = pd.read_csv('./data/train.csv')
test   = pd.read_csv('./data/test.csv')
layout = pd.read_csv('./data/layout_info.csv')

layout['layout_type_enc'] = layout['layout_type'].map(
    {v: i for i, v in enumerate(layout['layout_type'].unique())})
layout_feats = [c for c in layout.columns
                if c not in ['layout_id','layout_type','layout_type_enc']]
layout['layout_cluster'] = KMeans(n_clusters=8, random_state=SEED, n_init=10).fit_predict(
    StandardScaler().fit_transform(layout[layout_feats].fillna(0)))

train = train.merge(layout, on='layout_id', how='left')
test  = test.merge(layout,  on='layout_id', how='left')

for mask, df in [(train['layout_cluster'].isna(), train),
                 (test['layout_cluster'].isna(),  test)]:
    if mask.any():
        df.loc[mask, 'layout_cluster']  = 8
        df.loc[mask, 'layout_type_enc'] = -1

print(f'train: {train.shape}, test: {test.shape}')
print(f'unseen layouts - test: {test["layout_cluster"].isna().sum()}')


train: (250000, 110), test: (50000, 109)
unseen layouts - test: 0


## 2. 피처 엔지니어링

In [3]:
def engineer_features(df):
    d = df.copy()
    # 로봇 압박
    d['charge_pressure']       = d['charge_queue_length'] / (d['charger_count'] + 1)
    d['active_robot_ratio']    = d['robot_active'] / (d['robot_total'] + 1)
    d['idle_robot_ratio']      = d['robot_idle'] / (d['robot_total'] + 1)
    d['low_batt_robot_count']  = d['low_battery_ratio'] * d['robot_active']
    d['battery_stress']        = (1 - d['battery_mean'] / 100) * d['battery_std']
    d['effective_robot_avail'] = d['robot_total'] - d['low_batt_robot_count'] - d['charge_queue_length']
    d['robot_fault_ratio']     = d['fault_count_15m'] / (d['robot_active'] + 1)
    # 병목
    d['incident_total_15m']    = d['blocked_path_15m'] + d['near_collision_15m'] + d['fault_count_15m']
    d['congestion_load']       = d['congestion_score'] * d['avg_trip_distance']
    d['wait_per_intersection'] = d['intersection_wait_time_avg'] / (d['intersection_count'] + 1)
    d['path_congestion_gap']   = d['path_optimization_score'] - d['congestion_score']
    d['aisle_density']         = d['aisle_traffic_score'] / (d['aisle_width_avg'] + 0.1)
    # 주문 부하
    d['order_complexity']      = d['unique_sku_15m'] * d['avg_items_per_order']
    d['urgent_heavy_ratio']    = d['urgent_order_ratio'] * d['heavy_item_ratio']
    d['effective_order_load']  = d['order_inflow_15m'] * (1 + d['urgent_order_ratio'])
    d['rework_pressure']       = d['return_order_ratio'] + d['replenishment_overlap']
    d['pick_complexity']       = d['pick_list_length_avg'] * (1 - d['sku_concentration'])
    # 설비
    d['dock_pack_util_avg']      = (d['pack_utilization'] + d['loading_dock_util'] + d['staging_area_util']) / 3
    d['orders_per_pack_station'] = d['order_inflow_15m'] / (d['pack_station_count'] + 1)
    d['orders_per_robot']        = d['order_inflow_15m'] / (d['robot_active'] + 1)
    d['robot_density']           = d['robot_total'] / (d['floor_area_sqm'] + 1)
    d['pack_area_pressure']      = d['pack_utilization'] * d['layout_compactness']
    d['dock_truck_bottleneck']   = d['loading_dock_util'] * d['outbound_truck_wait_min']
    # IT
    d['it_bottleneck']         = d['wms_response_time_ms'] * (1 + d['scanner_error_rate'])
    d['barcode_fail_rate']     = 1 - d['barcode_read_success_rate']
    d['label_scan_bottleneck'] = d['label_print_queue'] * (1 + d['scanner_error_rate'])
    d['conveyor_load']         = d['avg_package_weight_kg'] / (d['conveyor_speed_mps'] + 0.01)
    # 인력
    d['orders_per_staff']      = d['order_inflow_15m'] / (d['staff_on_floor'] + 1)
    d['shift_load_ratio']      = d['order_inflow_15m'] / (d['prev_shift_volume'] + 1)
    d['handover_pressure']     = d['shift_handover_delay_min'] * d['prev_shift_volume'] / 1000
    # 시간
    d['shift_hour_sin'] = np.sin(2 * np.pi * d['shift_hour'] / 24)
    d['shift_hour_cos'] = np.cos(2 * np.pi * d['shift_hour'] / 24)
    d['dow_sin']        = np.sin(2 * np.pi * d['day_of_week'] / 7)
    d['dow_cos']        = np.cos(2 * np.pi * d['day_of_week'] / 7)
    d['is_peak_hour']   = d['shift_hour'].isin([9,10,11,14,15,16,17]).astype(int)
    # 레이아웃
    d['area_per_robot']          = d['floor_area_sqm'] / (d['robot_total'] + 1)
    d['crossdock_dock_pressure'] = d['cross_dock_ratio'] * d['loading_dock_util']
    d['robot_per_pack_station']  = d['robot_total'] / (d['pack_station_count'] + 1)
    # 교호작용
    d['robot_order_congestion']  = d['orders_per_robot'] * d['congestion_score']
    d['battery_order_pressure']  = d['battery_stress'] * d['effective_order_load']
    d['low_batt_x_congestion']   = d['low_battery_ratio'] * d['congestion_score']
    d['charge_q_x_orders']       = d['charge_queue_length'] * d['order_inflow_15m']
    # pack 임계점
    d['pack_saturated']         = (d['pack_utilization'] > 0.95).astype(int)
    d['pack_overflow_pressure'] = np.maximum(d['pack_utilization'] - 0.95, 0) * d['order_inflow_15m']
    d['pack_util_sq']           = d['pack_utilization'] ** 2
    # [v6] 비율 피처 (스케일 이동에 강건)
    d['order_robot_balance']   = d['order_inflow_15m'] / (d['robot_active'] * d['robot_utilization'] + 1)
    d['charge_demand_ratio']   = d['charge_queue_length'] / (d['robot_charging'] + 1)
    d['pack_order_ratio']      = d['pack_utilization'] / (d['order_inflow_15m'] / 100 + 0.01)
    d['congestion_per_robot']  = d['congestion_score'] / (d['robot_active'] + 1)
    d['fault_per_100_orders']  = d['fault_count_15m'] / (d['order_inflow_15m'] / 100 + 0.01)
    d['block_per_100_orders']  = d['blocked_path_15m'] / (d['order_inflow_15m'] / 100 + 0.01)
    d['dock_order_match']      = d['loading_dock_util'] / (d['order_inflow_15m'] / 200 + 0.01)
    d['staff_order_match']     = d['staff_on_floor'] / (d['order_inflow_15m'] / 50 + 0.01)
    d['battery_adequacy']      = d['battery_mean'] / (100 - d['low_battery_ratio'] * 100 + 1)
    d['trip_efficiency']       = d['avg_trip_distance'] / (d['robot_utilization'] + 0.01)
    return d

train_fe = engineer_features(train)
test_fe  = engineer_features(test)
print(f'피처 엔지니어링: {train_fe.shape}')


피처 엔지니어링: (250000, 165)


## 3. 래그 + 롤링 피처

In [4]:
LAG_COLS = [
    'order_inflow_15m','congestion_score','robot_utilization','battery_mean',
    'charge_queue_length','blocked_path_15m','fault_count_15m',
    'pack_utilization','loading_dock_util','effective_order_load','orders_per_robot',
]
CUM_COLS  = ['order_inflow_15m','congestion_score','blocked_path_15m','fault_count_15m','incident_total_15m']
ROLL_COLS = ['order_inflow_15m','congestion_score','battery_mean','pack_utilization','robot_utilization','loading_dock_util']

def add_temporal(df, lag_cols, cum_cols, roll_cols):
    d = df.copy().sort_values(['scenario_id','shift_hour']).reset_index(drop=True)
    d['slot_idx']      = d.groupby('scenario_id').cumcount()
    d['slot_progress'] = d['slot_idx'] / 24
    for col in lag_cols:
        if col not in d.columns: continue
        l1 = d.groupby('scenario_id')[col].shift(1)
        l2 = d.groupby('scenario_id')[col].shift(2)
        d[f'{col}_lag1']       = l1
        d[f'{col}_lag2']       = l2
        d[f'{col}_diff1']      = d[col] - l1
        d[f'{col}_pct_change'] = (d[col] - l1) / (l1.abs() + 1)
    for col in cum_cols:
        if col not in d.columns: continue
        d[f'{col}_cumsum'] = d.groupby('scenario_id')[col].cumsum()
    for col in roll_cols:
        if col not in d.columns: continue
        d[f'{col}_roll3_mean'] = d.groupby('scenario_id')[col].transform(
            lambda x: x.rolling(3, min_periods=1).mean())
        d[f'{col}_roll3_std']  = d.groupby('scenario_id')[col].transform(
            lambda x: x.rolling(3, min_periods=1).std())
    for col in ['order_inflow_15m','congestion_score','pack_utilization','battery_mean']:
        if col not in d.columns: continue
        d[f'{col}_exp_mean'] = d.groupby('scenario_id')[col].transform(
            lambda x: x.expanding().mean())
    for col in ['order_inflow_15m','congestion_score','pack_utilization']:
        if col not in d.columns: continue
        def rolling_slope(x):
            out = np.full(len(x), np.nan); vals = x.values
            for i in range(2, len(vals)):
                yv = vals[i-2:i+1]
                if np.all(np.isfinite(yv)): out[i] = (yv[2]-yv[0])/2
            return pd.Series(out, index=x.index)
        d[f'{col}_trend3'] = d.groupby('scenario_id')[col].transform(rolling_slope)
    if 'pack_utilization_lag1' in d.columns:
        d['pack_sat_lag1'] = (d['pack_utilization_lag1'] > 0.95).astype(int)
    if 'order_inflow_15m_cumsum' in d.columns:
        d['cum_order_per_robot'] = d['order_inflow_15m_cumsum'] / (d['robot_total'] + 1)
    for col in ['order_inflow_15m','congestion_score','pack_utilization']:
        rm = f'{col}_roll3_mean'
        if rm in d.columns:
            d[f'{col}_deviation']       = d[col] - d[rm]
            d[f'{col}_deviation_ratio'] = d[f'{col}_deviation'] / (d[rm].abs() + 1)
    fill_cols = [c for c in d.columns if any(t in c for t in
                 ['_lag','_diff','_cumsum','_roll3','_exp_','_trend','_sat_lag','_pct_change','_deviation'])]
    for col in fill_cols:
        d[col] = d.groupby('scenario_id')[col].transform(lambda x: x.fillna(x.median()))
    d[fill_cols] = d[fill_cols].fillna(0)
    return d

train_fe = add_temporal(train_fe, LAG_COLS, CUM_COLS, ROLL_COLS)
test_fe  = add_temporal(test_fe,  LAG_COLS, CUM_COLS, ROLL_COLS)
print(f'전체 피처: {len([c for c in train_fe.columns if c not in ID_COLS+[TARGET]])}')


전체 피처: 239


## 4. 시나리오 통계 (최소) + 타겟 인코딩

In [5]:
# 시나리오 통계 최소화 (과적합 방지)
for col in ['order_inflow_15m','congestion_score','robot_utilization',
            'battery_mean','pack_utilization','loading_dock_util']:
    if col not in train_fe.columns: continue
    for df in [train_fe, test_fe]:
        df[f'sc_{col}_mean'] = df.groupby('scenario_id')[col].transform('mean')
        df[f'sc_{col}_dev']  = df[col] - df[f'sc_{col}_mean']

# 타겟 인코딩 (alpha=50 보수적)
def target_encode_cv(tr, te, col, target, n_splits=5, alpha=50):
    gm = tr[target].mean(); enc = np.zeros(len(tr))
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=SEED)
    for ti, vi in kf.split(tr):
        s = tr.iloc[ti].groupby(col)[target].agg(['mean','count'])
        s['sm'] = (s['count']*s['mean']+alpha*gm)/(s['count']+alpha)
        enc[vi] = tr.iloc[vi][col].map(s['sm']).fillna(gm)
    fs = tr.groupby(col)[target].agg(['mean','count'])
    fs['sm'] = (fs['count']*fs['mean']+alpha*gm)/(fs['count']+alpha)
    return enc, te[col].map(fs['sm']).fillna(gm).values

for col_name, feat_name in [('layout_cluster','te_layout_cluster'),('layout_type_enc','te_layout_type')]:
    tr_e, te_e = target_encode_cv(train_fe, test_fe, col_name, TARGET)
    train_fe[feat_name] = tr_e; test_fe[feat_name] = te_e
print('타겟 인코딩 완료')


타겟 인코딩 완료


## 5. 피처 준비

In [6]:
feature_cols = [
    c for c in train_fe.columns
    if c not in ID_COLS + [TARGET]
    and pd.api.types.is_numeric_dtype(train_fe[c])
]
for col in feature_cols:
    med = train_fe[col].median()
    train_fe[col] = train_fe[col].fillna(med)
    test_fe[col]  = test_fe[col].fillna(med)

drop_cols = [c for c in feature_cols
             if train_fe[c].nunique() <= 1 or train_fe[c].std() < 1e-10]
feature_cols = [c for c in feature_cols if c not in drop_cols]
print(f'최종 피처: {len(feature_cols)}개 (제거: {len(drop_cols)}개)')

X      = train_fe[feature_cols].copy()
y      = train_fe[TARGET].copy()
X_test = test_fe[feature_cols].copy()

y_tr_sqrt  = np.sqrt(y.clip(lower=0))
y_tr_log   = np.log1p(y.clip(lower=0))
y_tr_power = np.power(y.clip(lower=0) + 1e-8, POWER)

def decode_sqrt(p):  return (p.clip(0))**2
def decode_log(p):   return np.expm1(p).clip(0)
def decode_power(p): return np.power(p.clip(0), 1/POWER)

gkf    = GroupKFold(n_splits=N_FOLDS)
groups = train_fe['scenario_id']
print(f'왜도: 원본={stats.skew(y):.2f}, sqrt={stats.skew(y_tr_sqrt):.2f}, log={stats.skew(y_tr_log):.2f}')


최종 피처: 252개 (제거: 0개)
왜도: 원본=5.68, sqrt=1.47, log=0.08


## 6. [v7 핵심] Adversarial Sample Weighting
train과 test 분포 차이(AUC=0.65)를 보정 — test처럼 보이는 train 샘플에 높은 가중치 부여

In [7]:
print('Adversarial 가중치 계산 중...')

X_adv = pd.concat([X, X_test], axis=0).reset_index(drop=True)
y_adv = np.array([0]*len(X) + [1]*len(X_test))

rf_adv = RandomForestClassifier(
    n_estimators=200, max_depth=5, random_state=SEED,
    n_jobs=-1, min_samples_leaf=50
)
# 5-fold cross_val_predict로 OOF P(test) 얻기
prob_test = cross_val_predict(
    rf_adv, X_adv, y_adv, cv=5, method='predict_proba'
)[:len(X), 1]  # train 부분의 P(test)

# Importance weight: P(test) / P(train) = p / (1-p)
adv_weights = prob_test / (1 - prob_test + 1e-6)
adv_weights = np.clip(adv_weights, 0.2, 5.0)  # 극단값 클리핑
adv_weights = adv_weights / adv_weights.mean()  # 정규화

adv_auc_approx = rf_adv.fit(X_adv, y_adv)
print(f'가중치 분포: min={adv_weights.min():.3f}, mean={adv_weights.mean():.3f}, max={adv_weights.max():.3f}')
print(f'가중치 > 1인 비율: {(adv_weights > 1).mean()*100:.1f}% (test와 유사한 train 샘플)')

# 상위 피처 (train/test 구분에 중요한 피처) 확인
imp_adv = pd.Series(rf_adv.feature_importances_, index=feature_cols).sort_values(ascending=False)
print('\n상위 10개 adversarial 피처 (이 피처들이 분포 차이 주범):')
for feat, val in imp_adv.head(10).items():
    print(f'  {feat:40s}: {val:.4f}')


Adversarial 가중치 계산 중...
가중치 분포: min=0.863, mean=1.000, max=4.443
가중치 > 1인 비율: 27.4% (test와 유사한 train 샘플)

상위 10개 adversarial 피처 (이 피처들이 분포 차이 주범):
  pack_station_count                      : 0.1141
  te_layout_type                          : 0.1101
  charger_count                           : 0.0972
  robot_total                             : 0.0968
  te_layout_cluster                       : 0.0657
  effective_robot_avail                   : 0.0541
  sc_order_inflow_15m_mean                : 0.0348
  sku_concentration                       : 0.0273
  ceiling_height_m                        : 0.0268
  order_inflow_15m_roll3_mean             : 0.0267


## 7. LGB Optuna 튜닝 (30 trials, adversarial weight 적용)

In [8]:
tidx = np.random.choice(len(X), 50000, replace=False)
Xt   = X.iloc[tidx].reset_index(drop=True)
yt_s = y_tr_sqrt.iloc[tidx].reset_index(drop=True)
yt_r = y.iloc[tidx].reset_index(drop=True)
gt   = groups.iloc[tidx].reset_index(drop=True)
wt   = adv_weights[tidx]  # adversarial 가중치

def lgb_obj(trial):
    p = dict(
        n_estimators      = trial.suggest_int('n_estimators', 500, 2500),
        learning_rate     = trial.suggest_float('learning_rate', 0.01, 0.08, log=True),
        max_depth         = trial.suggest_int('max_depth', 4, 8),
        num_leaves        = trial.suggest_int('num_leaves', 31, 200),
        subsample         = trial.suggest_float('subsample', 0.5, 0.9),
        colsample_bytree  = trial.suggest_float('colsample_bytree', 0.3, 0.8),
        reg_alpha         = trial.suggest_float('reg_alpha', 0.1, 50, log=True),
        reg_lambda        = trial.suggest_float('reg_lambda', 0.1, 50, log=True),
        min_child_samples = trial.suggest_int('min_child_samples', 50, 500),
    )
    F = dict(objective='mae', device='gpu', random_state=SEED, verbose=-1)
    ms = []
    for ti, vi in GroupKFold(n_splits=3).split(Xt, yt_s, groups=gt):
        m = LGBMRegressor(**p, **F)
        m.fit(Xt.iloc[ti], yt_s.iloc[ti],
              eval_set=[(Xt.iloc[vi], yt_s.iloc[vi])],
              sample_weight=wt[ti],
              callbacks=[lgb.early_stopping(30, verbose=False), lgb.log_evaluation(-1)])
        ms.append(mean_absolute_error(yt_r.iloc[vi], decode_sqrt(m.predict(Xt.iloc[vi]))))
    return np.mean(ms)

print('LGB Optuna (30 trials + adversarial weight)...')
study_lgb = optuna.create_study(direction='minimize',
                                 sampler=optuna.samplers.TPESampler(seed=SEED))
study_lgb.optimize(lgb_obj, n_trials=30, show_progress_bar=True)
BEST_LGB = study_lgb.best_params
print(f'LGB 최적 MAE: {study_lgb.best_value:.4f}')
print(f'파라미터: {BEST_LGB}')


LGB Optuna (30 trials + adversarial weight)...


Best trial: 10. Best value: 8.85199: 100%|██████████| 30/30 [22:46<00:00, 45.55s/it]

LGB 최적 MAE: 8.8520
파라미터: {'n_estimators': 2391, 'learning_rate': 0.010125488759446289, 'max_depth': 8, 'num_leaves': 95, 'subsample': 0.7144754354791039, 'colsample_bytree': 0.6036901462240624, 'reg_alpha': 0.10985676914755406, 'reg_lambda': 4.5288300780415165, 'min_child_samples': 71}


## 8. CB Optuna 튜닝 (20 trials, adversarial weight 적용)

In [9]:
def cb_obj(trial):
    p = dict(
        iterations       = trial.suggest_int('iterations', 500, 2500),
        learning_rate    = trial.suggest_float('learning_rate', 0.01, 0.08, log=True),
        depth            = trial.suggest_int('depth', 4, 8),
        l2_leaf_reg      = trial.suggest_float('l2_leaf_reg', 1.0, 50, log=True),
        subsample        = trial.suggest_float('subsample', 0.5, 0.9),
        min_data_in_leaf = trial.suggest_int('min_data_in_leaf', 50, 500),
    )
    F = dict(loss_function='MAE', eval_metric='MAE', bootstrap_type='Bernoulli',
             random_seed=SEED, early_stopping_rounds=30, verbose=0,
             task_type='GPU', devices='0')
    ms = []
    for ti, vi in GroupKFold(n_splits=3).split(Xt, yt_s, groups=gt):
        m = CatBoostRegressor(**p, **F)
        m.fit(Xt.iloc[ti], yt_s.iloc[ti],
              eval_set=(Xt.iloc[vi], yt_s.iloc[vi]),
              sample_weight=wt[ti])
        ms.append(mean_absolute_error(yt_r.iloc[vi], decode_sqrt(m.predict(Xt.iloc[vi]))))
    return np.mean(ms)

print('CB Optuna (20 trials + adversarial weight)...')
study_cb = optuna.create_study(direction='minimize',
                                sampler=optuna.samplers.TPESampler(seed=SEED))
study_cb.optimize(cb_obj, n_trials=20, show_progress_bar=True)
BEST_CB = study_cb.best_params
print(f'CB 최적 MAE: {study_cb.best_value:.4f}')
print(f'파라미터: {BEST_CB}')


CB Optuna (20 trials + adversarial weight)...


  0%|          | 0/20 [00:00<?, ?it/s]Default metric period is 5 because MAE is/are not implemented for GPU
Default metric period is 5 because MAE is/are not implemented for GPU
Default metric period is 5 because MAE is/are not implemented for GPU
Best trial: 0. Best value: 8.92035:   5%|▌         | 1/20 [01:13<23:19, 73.66s/it]Default metric period is 5 because MAE is/are not implemented for GPU
Default metric period is 5 because MAE is/are not implemented for GPU
Default metric period is 5 because MAE is/are not implemented for GPU
Best trial: 0. Best value: 8.92035:  10%|█         | 2/20 [02:08<18:43, 62.43s/it]Default metric period is 5 because MAE is/are not implemented for GPU
Default metric period is 5 because MAE is/are not implemented for GPU
Default metric period is 5 because MAE is/are not implemented for GPU
Best trial: 0. Best value: 8.92035:  15%|█▌        | 3/20 [03:15<18:21, 64.80s/it]Default metric period is 5 because MAE is/are not implemented for GPU
Default metric p

CB 최적 MAE: 8.9105
파라미터: {'iterations': 2001, 'learning_rate': 0.03965110640432585, 'depth': 8, 'l2_leaf_reg': 10.015848959884698, 'subsample': 0.6874820875551368, 'min_data_in_leaf': 129}


## 9. M1: LGB 5-Fold (sqrt) + Adversarial Weight

In [10]:
oof_lgb_sqrt, test_lgb_sqrt = np.zeros(len(X)), np.zeros(len(X_test))
lgb_imp = np.zeros(len(feature_cols))
print('M1: LGB(sqrt+adv_weight) 5-Fold')
for fold, (ti, vi) in enumerate(gkf.split(X, y, groups=groups)):
    print(f'-- Fold {fold+1} --')
    m = LGBMRegressor(**BEST_LGB, objective='mae', device='gpu', random_state=SEED, verbose=-1)
    m.fit(X.iloc[ti], y_tr_sqrt.iloc[ti],
          eval_set=[(X.iloc[vi], y_tr_sqrt.iloc[vi])],
          sample_weight=adv_weights[ti],
          callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(200)])
    oof_lgb_sqrt[vi] = decode_sqrt(m.predict(X.iloc[vi]))
    test_lgb_sqrt   += decode_sqrt(m.predict(X_test)) / N_FOLDS
    lgb_imp         += m.feature_importances_ / N_FOLDS
    print(f'   MAE: {mean_absolute_error(y.iloc[vi], oof_lgb_sqrt[vi]):.4f}')
print(f'M1 OOF MAE: {mean_absolute_error(y, oof_lgb_sqrt):.4f}')


M1: LGB(sqrt+adv_weight) 5-Fold
-- Fold 1 --
[200]	valid_0's l1: 0.949534
[400]	valid_0's l1: 0.921233
[600]	valid_0's l1: 0.91893
[800]	valid_0's l1: 0.91841
   MAE: 8.8152
-- Fold 2 --
[200]	valid_0's l1: 0.949362
[400]	valid_0's l1: 0.92232
[600]	valid_0's l1: 0.919981
[800]	valid_0's l1: 0.919507
   MAE: 8.7517
-- Fold 3 --
[200]	valid_0's l1: 0.991645
[400]	valid_0's l1: 0.969401
[600]	valid_0's l1: 0.967356
   MAE: 9.2881
-- Fold 4 --
[200]	valid_0's l1: 0.927353
[400]	valid_0's l1: 0.894542
[600]	valid_0's l1: 0.891316
[800]	valid_0's l1: 0.890542
   MAE: 8.3113
-- Fold 5 --
[200]	valid_0's l1: 0.93933
[400]	valid_0's l1: 0.910846
[600]	valid_0's l1: 0.907794
[800]	valid_0's l1: 0.907024
[1000]	valid_0's l1: 0.9066
   MAE: 8.6625
M1 OOF MAE: 8.7658


## 10. M2: CB 5-Fold (sqrt) + Adversarial Weight

In [11]:
oof_cb_sqrt, test_cb_sqrt = np.zeros(len(X)), np.zeros(len(X_test))
print('M2: CB(sqrt+adv_weight) 5-Fold')
for fold, (ti, vi) in enumerate(gkf.split(X, y, groups=groups)):
    print(f'-- Fold {fold+1} --')
    m = CatBoostRegressor(**BEST_CB, loss_function='MAE', eval_metric='MAE',
                          bootstrap_type='Bernoulli', random_seed=SEED,
                          early_stopping_rounds=50, verbose=200,
                          task_type='GPU', devices='0')
    m.fit(X.iloc[ti], y_tr_sqrt.iloc[ti],
          eval_set=(X.iloc[vi], y_tr_sqrt.iloc[vi]),
          sample_weight=adv_weights[ti])
    oof_cb_sqrt[vi] = decode_sqrt(m.predict(X.iloc[vi]))
    test_cb_sqrt   += decode_sqrt(m.predict(X_test)) / N_FOLDS
    print(f'   MAE: {mean_absolute_error(y.iloc[vi], oof_cb_sqrt[vi]):.4f}')
print(f'M2 OOF MAE: {mean_absolute_error(y, oof_cb_sqrt):.4f}')


M2: CB(sqrt+adv_weight) 5-Fold
-- Fold 1 --


Default metric period is 5 because MAE is/are not implemented for GPU


0:	learn: 1.7808090	test: 1.7210234	best: 1.7210234 (0)	total: 10.9ms	remaining: 21.9s
200:	learn: 0.9859894	test: 0.9627334	best: 0.9627334 (200)	total: 1.82s	remaining: 16.3s
400:	learn: 0.9330033	test: 0.9347850	best: 0.9347850 (400)	total: 3.58s	remaining: 14.3s
600:	learn: 0.8956855	test: 0.9307515	best: 0.9307303 (596)	total: 5.35s	remaining: 12.5s
bestTest = 0.9299785938
bestIteration = 691
Shrink model to first 692 iterations.
   MAE: 8.9231
-- Fold 2 --


Default metric period is 5 because MAE is/are not implemented for GPU


0:	learn: 1.7777214	test: 1.7103780	best: 1.7103780 (0)	total: 9.36ms	remaining: 18.7s
200:	learn: 0.9828101	test: 0.9587433	best: 0.9587433 (200)	total: 1.99s	remaining: 17.8s
400:	learn: 0.9291641	test: 0.9319201	best: 0.9319201 (400)	total: 3.93s	remaining: 15.7s
600:	learn: 0.8918951	test: 0.9250345	best: 0.9250345 (600)	total: 5.87s	remaining: 13.7s
bestTest = 0.923650625
bestIteration = 689
Shrink model to first 690 iterations.
   MAE: 8.7898
-- Fold 3 --


...

0:	learn: 1.7767317	test: 1.7264378	best: 1.7264378 (0)	total: 9.66ms	remaining: 19.3s
200:	learn: 0.9686900	test: 1.0165428	best: 1.0165428 (200)	total: 1.96s	remaining: 17.6s
400:	learn: 0.9178108	test: 0.9891424	best: 0.9891424 (400)	total: 3.88s	remaining: 15.5s
600:	learn: 0.8826259	test: 0.9825597	best: 0.9825597 (600)	total: 5.85s	remaining: 13.6s
800:	learn: 0.8501543	test: 0.9796016	best: 0.9796016 (800)	total: 7.83s	remaining: 11.7s
1000:	learn: 0.8239693	test: 0.9761347	best: 0.9760915 (996)	total: 9.88s	remaining: 9.87s
bestTest = 0.9753795312
bestIteration = 1081
Shrink model to first 1082 iterations.
   MAE: 9.3409
-- Fold 4 --


Default metric period is 5 because MAE is/are not implemented for GPU


0:	learn: 1.7852800	test: 1.7132667	best: 1.7132667 (0)	total: 10.2ms	remaining: 20.3s
200:	learn: 0.9936941	test: 0.9369112	best: 0.9369112 (200)	total: 1.92s	remaining: 17.2s
400:	learn: 0.9400304	test: 0.9036155	best: 0.9036155 (400)	total: 3.88s	remaining: 15.5s
600:	learn: 0.9033195	test: 0.8965652	best: 0.8965011 (599)	total: 5.89s	remaining: 13.7s
800:	learn: 0.8697734	test: 0.8942166	best: 0.8940784 (790)	total: 7.94s	remaining: 11.9s
bestTest = 0.8940783594
bestIteration = 790
Shrink model to first 791 iterations.
   MAE: 8.3315
-- Fold 5 --


Default metric period is 5 because MAE is/are not implemented for GPU


0:	learn: 1.7869174	test: 1.7166412	best: 1.7166412 (0)	total: 11.5ms	remaining: 23.1s
200:	learn: 0.9945297	test: 0.9429117	best: 0.9429117 (200)	total: 1.86s	remaining: 16.6s
400:	learn: 0.9403113	test: 0.9159287	best: 0.9159287 (400)	total: 3.59s	remaining: 14.3s
600:	learn: 0.9005976	test: 0.9114429	best: 0.9114033 (597)	total: 5.38s	remaining: 12.5s
800:	learn: 0.8664444	test: 0.9095662	best: 0.9093513 (775)	total: 7.21s	remaining: 10.8s
1000:	learn: 0.8378054	test: 0.9083643	best: 0.9082807 (991)	total: 9.05s	remaining: 9.04s
1200:	learn: 0.8107972	test: 0.9077617	best: 0.9077431 (1193)	total: 10.9s	remaining: 7.27s
bestTest = 0.907494375
bestIteration = 1225
Shrink model to first 1226 iterations.
   MAE: 8.6691
M2 OOF MAE: 8.8109


## 11. M3: XGB 5-Fold (sqrt)

In [12]:
oof_xgb_sqrt, test_xgb_sqrt = np.zeros(len(X)), np.zeros(len(X_test))
print('M3: XGB(sqrt) 5-Fold')
for fold, (ti, vi) in enumerate(gkf.split(X, y, groups=groups)):
    print(f'-- Fold {fold+1} --')
    m = XGBRegressor(
        n_estimators=2000, learning_rate=0.03, max_depth=6,
        subsample=0.7, colsample_bytree=0.5,
        reg_alpha=5, reg_lambda=5, min_child_weight=100,
        device='cuda', tree_method='hist',
        random_state=SEED, verbosity=0
    )
    m.fit(X.iloc[ti], y_tr_sqrt.iloc[ti],
          eval_set=[(X.iloc[vi], y_tr_sqrt.iloc[vi])],
          sample_weight=adv_weights[ti],
          verbose=200)
    oof_xgb_sqrt[vi] = decode_sqrt(m.predict(X.iloc[vi]))
    test_xgb_sqrt   += decode_sqrt(m.predict(X_test)) / N_FOLDS
    print(f'   MAE: {mean_absolute_error(y.iloc[vi], oof_xgb_sqrt[vi]):.4f}')
print(f'M3 OOF MAE: {mean_absolute_error(y, oof_xgb_sqrt):.4f}')


M3: XGB(sqrt) 5-Fold
-- Fold 1 --
[0]	validation_0-rmse:2.30212
[200]	validation_0-rmse:1.49710
[400]	validation_0-rmse:1.49252
[600]	validation_0-rmse:1.49167
[800]	validation_0-rmse:1.49280
[1000]	validation_0-rmse:1.49297
[1200]	validation_0-rmse:1.49355
[1400]	validation_0-rmse:1.49495
[1600]	validation_0-rmse:1.49714
[1800]	validation_0-rmse:1.49728
[1999]	validation_0-rmse:1.49794
   MAE: 8.9307
-- Fold 2 --
[0]	validation_0-rmse:2.29114
[200]	validation_0-rmse:1.50912
[400]	validation_0-rmse:1.50809
[600]	validation_0-rmse:1.50893
[800]	validation_0-rmse:1.50982
[1000]	validation_0-rmse:1.51246
[1200]	validation_0-rmse:1.51487
[1400]	validation_0-rmse:1.51638
[1600]	validation_0-rmse:1.51812
[1800]	validation_0-rmse:1.51938
[1999]	validation_0-rmse:1.52147
   MAE: 8.8920
-- Fold 3 --
[0]	validation_0-rmse:2.30342
[200]	validation_0-rmse:1.55770
[400]	validation_0-rmse:1.55778
[600]	validation_0-rmse:1.55931
[800]	validation_0-rmse:1.56177
[1000]	validation_0-rmse:1.56285
[1200]	

## 12. M4: LGB (log1p) + Adversarial Weight

In [13]:
oof_lgb_log, test_lgb_log = np.zeros(len(X)), np.zeros(len(X_test))
print('M4: LGB(log) 5-Fold')
for fold, (ti, vi) in enumerate(gkf.split(X, y, groups=groups)):
    m = LGBMRegressor(**BEST_LGB, objective='mae', device='gpu', random_state=SEED, verbose=-1)
    m.fit(X.iloc[ti], y_tr_log.iloc[ti],
          eval_set=[(X.iloc[vi], y_tr_log.iloc[vi])],
          sample_weight=adv_weights[ti],
          callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(-1)])
    oof_lgb_log[vi] = decode_log(m.predict(X.iloc[vi]))
    test_lgb_log   += decode_log(m.predict(X_test)) / N_FOLDS
    print(f'  Fold {fold+1} MAE: {mean_absolute_error(y.iloc[vi], oof_lgb_log[vi]):.4f}')
print(f'M4 OOF MAE: {mean_absolute_error(y, oof_lgb_log):.4f}')


M4: LGB(log) 5-Fold
  Fold 1 MAE: 8.8097
  Fold 2 MAE: 8.7300
  Fold 3 MAE: 9.2588
  Fold 4 MAE: 8.3143
  Fold 5 MAE: 8.6607
M4 OOF MAE: 8.7547


## 13. M5: CB (log1p)

In [14]:
oof_cb_log, test_cb_log = np.zeros(len(X)), np.zeros(len(X_test))
print('M5: CB(log) 5-Fold')
for fold, (ti, vi) in enumerate(gkf.split(X, y, groups=groups)):
    m = CatBoostRegressor(**BEST_CB, loss_function='MAE', eval_metric='MAE',
                          bootstrap_type='Bernoulli', random_seed=SEED,
                          early_stopping_rounds=50, verbose=0,
                          task_type='GPU', devices='0')
    m.fit(X.iloc[ti], y_tr_log.iloc[ti],
          eval_set=(X.iloc[vi], y_tr_log.iloc[vi]),
          sample_weight=adv_weights[ti])
    oof_cb_log[vi] = decode_log(m.predict(X.iloc[vi]))
    test_cb_log   += decode_log(m.predict(X_test)) / N_FOLDS
    print(f'  Fold {fold+1} MAE: {mean_absolute_error(y.iloc[vi], oof_cb_log[vi]):.4f}')
print(f'M5 OOF MAE: {mean_absolute_error(y, oof_cb_log):.4f}')


M5: CB(log) 5-Fold


Default metric period is 5 because MAE is/are not implemented for GPU


  Fold 1 MAE: 8.8810


Default metric period is 5 because MAE is/are not implemented for GPU


  Fold 2 MAE: 8.7631


...

  Fold 3 MAE: 9.2687


Default metric period is 5 because MAE is/are not implemented for GPU


  Fold 4 MAE: 8.3000


Default metric period is 5 because MAE is/are not implemented for GPU


  Fold 5 MAE: 8.6669
M5 OOF MAE: 8.7760


## 14. M6: LGB (power=0.4)

In [15]:
oof_lgb_pow, test_lgb_pow = np.zeros(len(X)), np.zeros(len(X_test))
print(f'M6: LGB(power={POWER}) 5-Fold')
for fold, (ti, vi) in enumerate(gkf.split(X, y, groups=groups)):
    m = LGBMRegressor(**BEST_LGB, objective='mae', device='gpu', random_state=SEED, verbose=-1)
    m.fit(X.iloc[ti], y_tr_power.iloc[ti],
          eval_set=[(X.iloc[vi], y_tr_power.iloc[vi])],
          sample_weight=adv_weights[ti],
          callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(-1)])
    oof_lgb_pow[vi] = decode_power(m.predict(X.iloc[vi]))
    test_lgb_pow   += decode_power(m.predict(X_test)) / N_FOLDS
    print(f'  Fold {fold+1} MAE: {mean_absolute_error(y.iloc[vi], oof_lgb_pow[vi]):.4f}')
print(f'M6 OOF MAE: {mean_absolute_error(y, oof_lgb_pow):.4f}')


M6: LGB(power=0.4) 5-Fold
  Fold 1 MAE: 8.8054
  Fold 2 MAE: 8.7379
  Fold 3 MAE: 9.2781
  Fold 4 MAE: 8.3106
  Fold 5 MAE: 8.6709
M6 OOF MAE: 8.7606


## 15. M7: Multi-seed LGB (5 seeds)

In [16]:
SEEDS = [42, 123, 456, 789, 2024]
oof_lgb_ms, test_lgb_ms = np.zeros(len(X)), np.zeros(len(X_test))
print(f'M7: LGB Multi-seed ({len(SEEDS)} seeds)')
for seed in SEEDS:
    oof_tmp, test_tmp = np.zeros(len(X)), np.zeros(len(X_test))
    for fold, (ti, vi) in enumerate(gkf.split(X, y, groups=groups)):
        m = LGBMRegressor(**BEST_LGB, objective='mae', device='gpu',
                          random_state=seed, verbose=-1,
                          subsample_seed=seed, feature_fraction_seed=seed)
        m.fit(X.iloc[ti], y_tr_sqrt.iloc[ti],
              eval_set=[(X.iloc[vi], y_tr_sqrt.iloc[vi])],
              sample_weight=adv_weights[ti],
              callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(-1)])
        oof_tmp[vi] = decode_sqrt(m.predict(X.iloc[vi]))
        test_tmp   += decode_sqrt(m.predict(X_test)) / N_FOLDS
    oof_lgb_ms  += oof_tmp  / len(SEEDS)
    test_lgb_ms += test_tmp / len(SEEDS)
    print(f'  Seed {seed}: OOF MAE = {mean_absolute_error(y, oof_tmp):.4f}')
print(f'M7 Multi-seed OOF MAE: {mean_absolute_error(y, oof_lgb_ms):.4f}')


M7: LGB Multi-seed (5 seeds)
  Seed 42: OOF MAE = 8.7592
  Seed 123: OOF MAE = 8.7642
  Seed 456: OOF MAE = 8.7639
  Seed 789: OOF MAE = 8.7620
  Seed 2024: OOF MAE = 8.7663
M7 Multi-seed OOF MAE: 8.7584


## 16. M8: ExtraTrees (다양성)

In [17]:
from sklearn.ensemble import ExtraTreesRegressor

oof_et, test_et = np.zeros(len(X)), np.zeros(len(X_test))
print('M8: ExtraTrees(sqrt) 5-Fold')
for fold, (ti, vi) in enumerate(gkf.split(X, y, groups=groups)):
    m = ExtraTreesRegressor(
        n_estimators=500, max_depth=15, min_samples_leaf=20,
        max_features=0.5, random_state=SEED, n_jobs=-1
    )
    m.fit(X.iloc[ti], y_tr_sqrt.iloc[ti],
          sample_weight=adv_weights[ti])
    oof_et[vi] = decode_sqrt(m.predict(X.iloc[vi]))
    test_et   += decode_sqrt(m.predict(X_test)) / N_FOLDS
    print(f'  Fold {fold+1} MAE: {mean_absolute_error(y.iloc[vi], oof_et[vi]):.4f}')
print(f'M8 OOF MAE: {mean_absolute_error(y, oof_et):.4f}')


M8: ExtraTrees(sqrt) 5-Fold
  Fold 1 MAE: 8.9111
  Fold 2 MAE: 8.8743
  Fold 3 MAE: 9.3794
  Fold 4 MAE: 8.3063
  Fold 5 MAE: 8.7904
M8 OOF MAE: 8.8523


## 17. 앙상블 + Ridge 스태킹

In [18]:
oofs  = [oof_lgb_sqrt, oof_cb_sqrt, oof_xgb_sqrt, oof_lgb_log,
          oof_cb_log, oof_lgb_pow, oof_lgb_ms, oof_et]
tests = [test_lgb_sqrt, test_cb_sqrt, test_xgb_sqrt, test_lgb_log,
          test_cb_log, test_lgb_pow, test_lgb_ms, test_et]
names = ['LGB(sqrt)','CB(sqrt)','XGB(sqrt)','LGB(log)',
          'CB(log)','LGB(pow)','LGB(ms)','ExtraTrees']
n_m   = len(oofs)

print('=== 개별 모델 OOF MAE ===')
for name, oof in zip(names, oofs):
    print(f'  {name:>15}: {mean_absolute_error(y, oof):.4f}')

# SLSQP 가중합
def ens_loss(w): return mean_absolute_error(y, sum(w[i]*oofs[i] for i in range(n_m)))
res = minimize(ens_loss, [1/n_m]*n_m, method='SLSQP',
               bounds=[(0,1)]*n_m,
               constraints={'type':'eq','fun':lambda w:sum(w)-1})
weights_slsqp = res.x
print(f'\nSLSQP OOF MAE: {res.fun:.4f}')
for n, w in zip(names, weights_slsqp):
    if w > 0.01: print(f'  {n:>15}: w={w:.3f}')

# Ridge 스태킹 (auto alpha)
oof_stack  = np.column_stack(oofs)
test_stack = np.column_stack(tests)
best_alpha, best_mae_r = None, np.inf
for alpha in [0.1, 1.0, 10.0, 100.0]:
    oof_tmp = np.zeros(len(y))
    for fold, (ti, vi) in enumerate(gkf.split(X, y, groups=groups)):
        meta = Ridge(alpha=alpha)
        meta.fit(oof_stack[ti], y.iloc[ti])
        oof_tmp[vi] = meta.predict(oof_stack[vi])
    mae_r = mean_absolute_error(y, oof_tmp.clip(0))
    print(f'  Ridge(alpha={alpha:6.1f}): {mae_r:.4f}')
    if mae_r < best_mae_r: best_mae_r, best_alpha = mae_r, alpha

oof_ridge, test_ridge = np.zeros(len(y)), np.zeros(len(X_test))
for fold, (ti, vi) in enumerate(gkf.split(X, y, groups=groups)):
    meta = Ridge(alpha=best_alpha)
    meta.fit(oof_stack[ti], y.iloc[ti])
    oof_ridge[vi] = meta.predict(oof_stack[vi])
    test_ridge   += meta.predict(test_stack) / N_FOLDS
print(f'Ridge(alpha={best_alpha}) OOF MAE: {mean_absolute_error(y, oof_ridge.clip(0)):.4f}')


=== 개별 모델 OOF MAE ===
        LGB(sqrt): 8.7658
         CB(sqrt): 8.8109
        XGB(sqrt): 8.9060
         LGB(log): 8.7547
          CB(log): 8.7760
         LGB(pow): 8.7606
          LGB(ms): 8.7584
       ExtraTrees: 8.8523

SLSQP OOF MAE: 8.7184
        XGB(sqrt): w=0.186
         LGB(log): w=0.408
          CB(log): w=0.297
       ExtraTrees: w=0.109
  Ridge(alpha=   0.1): 9.2776
  Ridge(alpha=   1.0): 9.2776
  Ridge(alpha=  10.0): 9.2776
  Ridge(alpha= 100.0): 9.2775
Ridge(alpha=100.0) OOF MAE: 9.2775


## 18. 최종 앙상블 + 강화된 Quantile Mapping

In [19]:
oof_slsqp = sum(weights_slsqp[i]*oofs[i] for i in range(n_m)).clip(0)
mae_slsqp = mean_absolute_error(y, oof_slsqp)
mae_ridge = mean_absolute_error(y, oof_ridge.clip(0))
print(f'SLSQP: {mae_slsqp:.4f}, Ridge: {mae_ridge:.4f}')

if mae_ridge < mae_slsqp:
    oof_final = oof_ridge.clip(0)
    raw_test  = test_ridge.clip(0)
    print('=> Ridge 선택')
else:
    oof_final = oof_slsqp
    raw_test  = sum(weights_slsqp[i]*tests[i] for i in range(n_m)).clip(0)
    print('=> SLSQP 선택')

# Quantile Mapping (tail 강화)
qs = np.arange(0.01, 1.00, 0.01)
q_true = np.quantile(y, qs)
q_pred = np.quantile(oof_final, qs)

print('\n분위수 비교 (실제 vs 예측):')
for p in [.50,.75,.90,.95,.99]:
    i = int(p*100)-1
    print(f'  {p:.0%}: 실제={q_true[i]:.1f}, 예측={q_pred[i]:.1f}')

oof_mapped = np.interp(oof_final, q_pred, q_true)
mae_bef = mean_absolute_error(y, oof_final)
mae_aft = mean_absolute_error(y, oof_mapped)
print(f'\nQM: {mae_bef:.4f} => {mae_aft:.4f} ({mae_aft-mae_bef:+.4f})')

if mae_aft < mae_bef:
    final_preds = np.interp(raw_test, q_pred, q_true).clip(0)
    print('QM 적용')
else:
    final_preds = raw_test
    print('QM 미적용')

print(f'\n최종 예측 분포:')
print(f'  mean={final_preds.mean():.2f}, std={final_preds.std():.2f}')
print(f'  95%={np.percentile(final_preds,95):.1f}, 99%={np.percentile(final_preds,99):.1f}, max={final_preds.max():.1f}')


SLSQP: 8.7184, Ridge: 9.2775
=> SLSQP 선택

분위수 비교 (실제 vs 예측):
  50%: 실제=9.0, 예측=8.7
  75%: 실제=25.8, 예측=30.7
  90%: 실제=45.2, 예측=36.0
  95%: 실제=60.8, 예측=38.3
  99%: 실제=120.9, 예측=45.4

QM: 8.7184 => 10.1218 (+1.4034)
QM 미적용

최종 예측 분포:
  mean=19.26, std=14.24
  95%=39.9, 99%=47.6, max=76.0


## 19. Feature Importance

In [20]:
imp_df = pd.DataFrame({'feature':feature_cols,'importance':lgb_imp}).sort_values('importance',ascending=False).reset_index(drop=True)
print('=== Top 30 Features ===')
for _, row in imp_df.head(30).iterrows():
    print(f"  {row['feature']:45s} {row['importance']:8.1f}")


=== Top 30 Features ===
  sc_pack_utilization_mean                        3209.0
  sc_loading_dock_util_mean                       2906.2
  sc_congestion_score_mean                        2884.8
  sc_order_inflow_15m_mean                        2796.2
  sc_robot_utilization_mean                       2695.4
  sc_battery_mean_mean                            2512.8
  avg_trip_distance                               2406.6
  layout_compactness                              1902.8
  sku_concentration                               1714.8
  ceiling_height_m                                1671.4
  zone_dispersion                                 1666.6
  robot_per_pack_station                          1664.0
  aisle_width_avg                                 1417.2
  fire_sprinkler_count                            1414.0
  intersection_count                              1353.4
  one_way_ratio                                   1349.6
  floor_area_sqm                                  1290.6
  pack_

## 20. 제출

In [21]:
sub = pd.read_csv('./data/sample_submission.csv').drop(columns=[TARGET], errors='ignore')
sub = sub.merge(pd.DataFrame({'ID':test_fe['ID'], TARGET:final_preds}), on='ID', how='left')
sub.to_csv('./submission_v7.csv', index=False)
print('submission_v7.csv 저장 완료')
print(sub[TARGET].describe())
print(f'  v6 max: 71.1 -> v7 max: {final_preds.max():.1f} (차이: {final_preds.max()-71.1:+.1f})')


submission_v7.csv 저장 완료
count    50000.000000
mean        19.263692
std         14.240375
min          0.473417
25%          5.488403
50%         14.632579
75%         33.777725
max         76.029313
Name: avg_delay_minutes_next_30m, dtype: float64
  v6 max: 71.1 -> v7 max: 76.0 (차이: +4.9)
